In [1]:
import os, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import mlflow, mlflow.sklearn
from dotenv import load_dotenv
from mlflow.tracking import MlflowClient
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, recall_score, precision_score,
                             roc_auc_score, ConfusionMatrixDisplay)
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

load_dotenv("../.env")
print(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment("ids-binaire")

https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow


2026/11/20 00:31:14 INFO mlflow.tracking.fluent: Experiment with name 'ids-binaire' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/45afeab36d4141de92fa72dba1db8869', creation_time=1795131069499, experiment_id='0', last_update_time=1795131069499, lifecycle_stage='active', name='ids-binaire', tags={}>

In [2]:
df = pd.read_parquet("../data/processed/flows_clean.parquet")
FEATURES = [c for c in df.columns if c not in ("Label", "is_attack")]
X = df[FEATURES].astype("float32")
y = df["is_attack"]
print(len(FEATURES), "colonnes")

# sauvegarder la liste exacte et l'ordre des colonnes (pour le streaming)
os.makedirs("../models", exist_ok=True)
with open("../models/feature_columns.json", "w") as f:
    json.dump(FEATURES, f)

77 colonnes


In [3]:
# 70 % entraînement, 15 % validation, 15 % test
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, stratify=y_tmp, random_state=42)
print(X_train.shape, X_val.shape, X_test.shape)

(1539394, 77) (329937, 77) (329882, 77)


In [4]:
def run(nom, model, params, X_tr, y_tr, fit_kwargs=None):
    with mlflow.start_run(run_name=nom) as r:
        mlflow.log_params({"model": nom, "n_train_rows": len(X_tr), **params})

        t = time.time()
        model.fit(X_tr, y_tr, **(fit_kwargs or {}))
        duree = time.time() - t

        pred = model.predict(X_val)
        proba = model.predict_proba(X_val)[:, 1]
        metrics = {
            "f1_macro": f1_score(y_val, pred, average="macro"),
            "recall_attack": recall_score(y_val, pred),
            "precision_attack": precision_score(y_val, pred),
            "roc_auc": roc_auc_score(y_val, proba),
            "fit_seconds": duree,
        }
        mlflow.log_metrics(metrics)

        fig, ax = plt.subplots(figsize=(4, 4))
        ConfusionMatrixDisplay.from_predictions(y_val, pred, ax=ax, colorbar=False)
        mlflow.log_figure(fig, "confusion_matrix.png")
        plt.close(fig)

        mlflow.log_artifact("../models/feature_columns.json")
        mlflow.sklearn.log_model(model, artifact_path="model")
        print(nom, {k: round(v, 4) for k, v in metrics.items()})
        return r.info.run_id

In [5]:
# 1) LightGBM (rapide, très bon)
run("lightgbm",
    LGBMClassifier(n_estimators=300, learning_rate=0.1,
                   class_weight="balanced", n_jobs=-1, random_state=42),
    {"n_estimators": 300, "learning_rate": 0.1, "class_weight": "balanced"},
    X_train, y_train)

# 2) XGBoost : pas de class_weight, on utilise scale_pos_weight (même idée)
spw = float((y_train == 0).sum() / (y_train == 1).sum())
run("xgboost",
    XGBClassifier(n_estimators=300, learning_rate=0.1, tree_method="hist",
                  scale_pos_weight=spw, n_jobs=-1, random_state=42),
    {"n_estimators": 300, "learning_rate": 0.1, "scale_pos_weight": round(spw, 2)},
    X_train, y_train)

# 3) RandomForest : lent sur 1,5 million de lignes → on entraîne sur un échantillon
X_rf, _, y_rf, _ = train_test_split(
    X_train, y_train, train_size=400_000, stratify=y_train, random_state=42)
run("random_forest",
    RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=5,
                           class_weight="balanced", n_jobs=-1, random_state=42),
    {"n_estimators": 100, "max_depth": 20, "min_samples_leaf": 5,
     "class_weight": "balanced"},
    X_rf, y_rf)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 235536, number of negative: 1303858
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.150832 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14094
[LightGBM] [Info] Number of data points in the train set: 1539394, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


2026/11/20 00:37:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


lightgbm {'f1_macro': 0.9944, 'recall_attack': 0.9917, 'precision_attack': 0.9893, 'roc_auc': 0.999, 'fit_seconds': 30.1427}
🏃 View run lightgbm at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/0/runs/9fa0e6925d8c431ba267e6d2e45a5186
🧪 View experiment at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/0


2026/11/20 00:38:25 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


xgboost {'f1_macro': 0.9983, 'recall_attack': 0.9995, 'precision_attack': 0.9949, 'roc_auc': 1.0, 'fit_seconds': 58.1783}
🏃 View run xgboost at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/0/runs/c5075b8c3fac422eb4ef9b4d92c4d42e
🧪 View experiment at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/0


2026/11/20 00:39:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


random_forest {'f1_macro': 0.9968, 'recall_attack': 0.9955, 'precision_attack': 0.9937, 'roc_auc': 0.9999, 'fit_seconds': 30.6939}
🏃 View run random_forest at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/0/runs/bbcd8051121c4c4283f54bfa77062b26
🧪 View experiment at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/0


'bbcd8051121c4c4283f54bfa77062b26'

In [6]:
runs = mlflow.search_runs(experiment_names=["ids-binaire"],
                          order_by=["metrics.f1_macro DESC"], max_results=5)
print(runs[["tags.mlflow.runName", "metrics.f1_macro",
            "metrics.recall_attack", "run_id"]])
RUN_ID = runs.loc[0, "run_id"]

  tags.mlflow.runName  metrics.f1_macro  metrics.recall_attack  \
0             xgboost          0.998343               0.999485   
1       random_forest          0.996822               0.995523   
2            lightgbm          0.994404               0.991720   

                             run_id  
0  c5075b8c3fac422eb4ef9b4d92c4d42e  
1  bbcd8051121c4c4283f54bfa77062b26  
2  9fa0e6925d8c431ba267e6d2e45a5186  


In [7]:
best = mlflow.sklearn.load_model(f"runs:/{RUN_ID}/model")
pred = best.predict(X_test)
test_metrics = {
    "test_f1_macro": f1_score(y_test, pred, average="macro"),
    "test_recall_attack": recall_score(y_test, pred),
    "test_precision_attack": precision_score(y_test, pred),
}
print(test_metrics)
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metrics(test_metrics)

c:\Users\User\Documents\mlops-network-intrusion-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'test_f1_macro': 0.998126520897695, 'test_recall_attack': 0.9992075127788564, 'test_precision_attack': 0.9944592329685498}
🏃 View run xgboost at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/0/runs/c5075b8c3fac422eb4ef9b4d92c4d42e
🧪 View experiment at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/0


In [8]:
mv = mlflow.register_model(f"runs:/{RUN_ID}/model", "ids-binaire")
client = MlflowClient()
client.set_registered_model_alias("ids-binaire", "champion", mv.version)
print("version", mv.version, "→ champion")

Successfully registered model 'ids-binaire'.
2026/11/20 00:49:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ids-binaire, version 1
Created version '1' of model 'ids-binaire'.


version 1 → champion


In [9]:
m = mlflow.sklearn.load_model("models:/ids-binaire@champion")
print(m.predict(X_test.iloc[:5]))

[1 0 0 1 0]
